# BERTopic: Topic Evolution Quality (TTC, TTS, TTQ)

**Scenario 2** — Evaluates whether topic transitions are smooth and coherent.
- **TTC**: C_v coherence of topic words at t against corpus at t+1
- **TTS**: Vocabulary overlap between t and t+1
- **TTQ**: Harmonic mean of TTC and TTS

In [1]:
import ast
import time
import pandas as pd
import numpy as np
from pathlib import Path
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings
warnings.filterwarnings("ignore")

In [2]:
MODEL_NAME = "bertopic"
LIST_SUBJECT = ["cs", "math", "physics"]
DATA_DIR = Path("../../../../data/preprocess")
TEMPORAL_DIR = Path("../../../../results/bertopic/temporal")
RESULT_DIR = Path("../../../../results/bertopic/evolution")
DATA_FORMAT = "emb"
VERSION = "v1"
TOP_N = 10

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def parse_text(x):
    return str(x).split()


def compute_tts(words_t, words_next):
    """Temporal Topic Smoothness: overlap ratio between t and t+1."""
    if not words_t:
        return 0.0
    return len(set(words_t) & set(words_next)) / len(set(words_t))


def harmonic_mean(a, b):
    """F1-style harmonic mean."""
    if (a + b) <= 0:
        return 0.0
    return 2 * a * b / (a + b)

## Compute TTC, TTS, TTQ per transition

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Topic Evolution: {subject.upper()} (BERTopic)")
    print(f"{'='*70}")

    # Load topic word evolution from Scenario 1
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    # Load per-year coherence from Scenario 1 (baseline for ratio)
    metrics_df = pd.read_csv(TEMPORAL_DIR / subject / "per_year_metrics.csv")
    year_coherence = dict(zip(metrics_df["year"], metrics_df["coherence_cv"]))

    # Load text data for coherence evaluation
    data_df = pd.read_csv(DATA_DIR / subject / DATA_FORMAT / f"{VERSION}.csv")
    data_df["year"] = pd.to_datetime(data_df["submitted_date"]).dt.year

    # Parse topic words per (year, topic_id)
    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        words = [w.strip() for w in str(row["top_words"]).split(",")][:TOP_N]
        topic_words[key] = words

    years = sorted(data_df["year"].unique())
    all_topics = sorted(evo_df["topic_id"].unique())

    all_rows = []
    transition_rows = []

    for i in range(len(years) - 1):
        t, t_next = int(years[i]), int(years[i + 1])
        start = time.time()

        # Build corpus for t+1 (target for cross-time coherence)
        texts_next = [parse_text(x) for x in data_df[data_df["year"] == t_next]["text"]]
        dict_next = Dictionary(texts_next)

        # Find topics valid at both t and t+1
        valid_tids = [tid for tid in all_topics
                      if (t, tid) in topic_words and (t_next, tid) in topic_words
                      and len(topic_words[(t, tid)]) >= 2]

        # Filter words to only those in dict (avoid OOV)
        filtered_tids = []
        filtered_words_t = []
        for tid in valid_tids:
            in_vocab = [w for w in topic_words[(t, tid)] if w in dict_next.token2id]
            if len(in_vocab) >= 2:
                filtered_tids.append(tid)
                filtered_words_t.append(in_vocab)
                # print(f"tid {tid}: {len(in_vocab)} words")

        # Batch TTC: coherence of words at t against corpus at t+1
        if filtered_words_t:
            cm = CoherenceModel(topics=filtered_words_t, texts=texts_next,
                                dictionary=dict_next, coherence='c_v', processes=1)
            per_topic_coh = cm.get_coherence_per_topic()
        else:
            per_topic_coh = []

        # Get baseline coherence at time t from Scenario 1 results
        coherence_at_t_next = year_coherence.get(t_next, 0.0)

        # Per-topic metrics
        ttc_list, tts_list, ttq_list = [], [], []
        for idx, tid in enumerate(filtered_tids):
            tts = compute_tts(topic_words[(t, tid)], topic_words[(t_next, tid)])
            ttc = per_topic_coh[idx] if idx < len(per_topic_coh) else 0.0
            ttq = harmonic_mean(ttc, tts)

            ttc_list.append(ttc)
            tts_list.append(tts)
            ttq_list.append(ttq)

            all_rows.append({"subject": subject, "year_from": t, "year_to": t_next,
                             "topic_id": int(tid),
                             "ttc": round(ttc, 6), "tts": round(tts, 6), "ttq": round(ttq, 6)})

        avg_ttc = np.mean(ttc_list) if ttc_list else 0.0
        avg_tts = np.mean(tts_list) if tts_list else 0.0
        avg_ttq = np.mean(ttq_list) if ttq_list else 0.0
        ttc_ratio = avg_ttc / coherence_at_t_next if coherence_at_t_next > 0 else 0.0
        elapsed = time.time() - start

        transition_rows.append({"subject": subject, "year_from": t, "year_to": t_next,
                                "n_topics": len(filtered_tids),
                                "avg_ttc": round(avg_ttc, 6), "coherence_at_t_next": round(coherence_at_t_next, 6),
                                "ttc_ratio": round(ttc_ratio, 6),
                                "avg_tts": round(avg_tts, 6),
                                "avg_ttq": round(avg_ttq, 6)})

        print(f"  {t}→{t_next}: TTC={avg_ttc:.4f}  C(t+1)={coherence_at_t_next:.4f}  "
              f"ratio={ttc_ratio:.2%}  TTS={avg_tts:.4f}  "
              f"TTQ={avg_ttq:.4f}  ({len(filtered_tids)} topics) [{elapsed:.1f}s]")

    # Save per-topic metrics
    pd.DataFrame(all_rows).to_csv(
        RESULT_DIR / subject / "topic_evolution_metrics.csv", index=False)

    # Save transition summary
    trans_df = pd.DataFrame(transition_rows)
    trans_df.to_csv(RESULT_DIR / subject / "transition_summary.csv", index=False)

    # Overall summary
    subj_trans = trans_df[trans_df["subject"] == subject]
    overall = {"subject": subject,
               "avg_ttc": round(subj_trans["avg_ttc"].mean(), 6),
               "std_ttc": round(subj_trans["avg_ttc"].std(), 6),
               "avg_tts": round(subj_trans["avg_tts"].mean(), 6),
               "std_tts": round(subj_trans["avg_tts"].std(), 6),
               "avg_ttq": round(subj_trans["avg_ttq"].mean(), 6),
               "std_ttq": round(subj_trans["avg_ttq"].std(), 6),
               "avg_ttc_ratio": round(subj_trans["ttc_ratio"].mean(), 6),
               "std_ttc_ratio": round(subj_trans["ttc_ratio"].std(), 6)}
    pd.DataFrame([overall]).to_csv(
        RESULT_DIR / subject / "evolution_summary.csv", index=False)

    print(f"\n  Overall: TTC={overall['avg_ttc']:.4f}±{overall['std_ttc']:.4f}  "
          f"TTS={overall['avg_tts']:.4f}±{overall['std_tts']:.4f}  "
          f"TTQ={overall['avg_ttq']:.4f}±{overall['std_ttq']:.4f}  "
          f"ratio={overall['avg_ttc_ratio']:.2%}±{overall['std_ttc_ratio']:.2%}")
    print(f"  Saved to: {RESULT_DIR / subject}")


Topic Evolution: CS (BERTopic)
  2000→2001: TTC=0.4210  C(t+1)=0.6776  ratio=62.13%  TTS=0.1861  TTQ=0.2185  (36 topics) [0.3s]
  2001→2002: TTC=0.3908  C(t+1)=0.6816  ratio=57.33%  TTS=0.2154  TTQ=0.2390  (39 topics) [0.3s]
  2002→2003: TTC=0.4245  C(t+1)=0.6770  ratio=62.71%  TTS=0.2575  TTQ=0.2771  (40 topics) [0.4s]
  2003→2004: TTC=0.3855  C(t+1)=0.6277  ratio=61.41%  TTS=0.2280  TTQ=0.2501  (50 topics) [0.4s]
  2004→2005: TTC=0.3962  C(t+1)=0.6486  ratio=61.09%  TTS=0.2446  TTQ=0.2616  (56 topics) [0.5s]
  2005→2006: TTC=0.4433  C(t+1)=0.6451  ratio=68.71%  TTS=0.2611  TTQ=0.2778  (54 topics) [0.6s]
  2006→2007: TTC=0.4072  C(t+1)=0.6007  ratio=67.78%  TTS=0.2746  TTQ=0.2827  (59 topics) [0.6s]
  2007→2008: TTC=0.4111  C(t+1)=0.6292  ratio=65.33%  TTS=0.2790  TTQ=0.2939  (62 topics) [0.6s]
  2008→2009: TTC=0.4032  C(t+1)=0.6231  ratio=64.70%  TTS=0.2690  TTQ=0.2854  (71 topics) [0.7s]
  2009→2010: TTC=0.4195  C(t+1)=0.5780  ratio=72.57%  TTS=0.3038  TTQ=0.3136  (79 topics) [1.0s

## Evolution Summary

In [5]:
print(f"\n{'='*70}")
print(f"BERTopic — Evolution Summary")
print(f"{'='*70}")

for subject in LIST_SUBJECT:
    summary = pd.read_csv(RESULT_DIR / subject / "evolution_summary.csv").iloc[0]
    trans = pd.read_csv(RESULT_DIR / subject / "transition_summary.csv")

    print(f"\n  {subject.upper()}:")
    print(f"    TTC = {summary['avg_ttc']:.4f} ± {summary['std_ttc']:.4f}")
    print(f"    TTS = {summary['avg_tts']:.4f} ± {summary['std_tts']:.4f}")
    print(f"    TTQ = {summary['avg_ttq']:.4f} ± {summary['std_ttq']:.4f}")
    print(f"    TTQ range: {trans['avg_ttq'].min():.4f} — {trans['avg_ttq'].max():.4f}")


BERTopic — Evolution Summary

  CS:
    TTC = 0.4600 ± 0.0765
    TTS = 0.4073 ± 0.1831
    TTQ = 0.3950 ± 0.1479
    TTQ range: 0.2185 — 0.6710

  MATH:
    TTC = 0.4594 ± 0.0621
    TTS = 0.5040 ± 0.1501
    TTQ = 0.4573 ± 0.1160
    TTQ range: 0.2813 — 0.6304

  PHYSICS:
    TTC = 0.4661 ± 0.0625
    TTS = 0.4314 ± 0.1463
    TTQ = 0.4196 ± 0.1209
    TTQ range: 0.2521 — 0.6007
